# Baseline — najprostsze podejście (wersja 0)

Najnaiwniejszy sposób: **wszystkie akta w jeden prompt** + polecenie „jesteś prawnikiem, napisz apelację". Bez planu, bez podziału na kroki, bez selekcji dokumentów.

Ten notebook: generuje apelację baseline i **ewaluuje** ją (pokrycie + halucynacje), żeby było co porównać z agentem liniowym (`linear_walkthrough.ipynb`) i planerem (`planner_walkthrough.ipynb`).

> Odpowiednik `baseline/main.py`, ale interaktywnie.

## 0. Konfiguracja

Konfiguracja LLM (klucz, model) jest w pliku `.env`. **Baseline wymaga modelu
z dużym oknem kontekstu** (prompt ~19k tokenów) — użyj OpenAI; lokalna Ollama
domyślnie obetnie kontekst do 4096.

In [1]:
import os

# Wczytaj konfigurację LLM z pliku .env (klucz nie jest wpisywany w komórce).
from dotenv import load_dotenv

load_dotenv()

print("model:", os.environ.get("LLM_MODEL"))
print("klucz z .env wczytany:", bool(os.environ.get("LLM_API_KEY")))

model: gpt-5.4-mini
klucz z .env wczytany: True


## 1. Wczytanie akt i rozmiar promptu

Baseline wrzuca **całość** w jeden prompt. Policzmy, ile to tokenów — to sedno problemu tego podejścia.

In [2]:
from src.loader import load_all
from src.tokens import count_tokens
from baseline.main import build_context
from baseline.prompts import SYSTEM_PROMPT

docs = load_all()
context = build_context(docs)
prompt_tokens = count_tokens(SYSTEM_PROMPT + "\n\n" + context)

print(f"Dokumentów: {len(docs)}")
print(f"Znaków w kontekście: {len(context):,}")
print(f"Tokenów w promptcie (wejście): ~{prompt_tokens:,}")

Dokumentów: 16
Znaków w kontekście: 52,159
Tokenów w promptcie (wejście): ~19,341


## 2. Generowanie apelacji (jeden prompt)

Jedno wywołanie LLM — całe akta wchodzą, wychodzi gotowa apelacja.

In [11]:
from baseline.main import generate_appeal

appeal = generate_appeal(docs).tekst
print(f"Apelacja: {len(appeal):,} znaków\n")
print(appeal[:1000], "...")

Apelacja: 7,911 znaków

Sąd Apelacyjny w Krakowie
za pośrednictwem
Sądu Rejonowego dla Krakowa–Krowodrzy w Krakowie
II Wydział Karny

Sygn. akt II K 25/25

OSKARŻONY: Daniel Dzik
obrońca: radca prawny Lidia Lis

APELACJA
od wyroku Sądu Rejonowego dla Krakowa–Krowodrzy w Krakowie
z dnia 14 marca 2025 r., sygn. akt II K 25/25

Działając w imieniu oskarżonego Daniela Dzika, na podstawie art. 425 § 1 i 2 k.p.k., art. 444 k.p.k. oraz art. 427 § 1 i 2 k.p.k., zaskarżam powyższy wyrok w całości, tj. w zakresie punktów 1–9.

Zaskarżonemu wyrokowi zarzucam:

I. na podstawie art. 438 pkt 2 k.p.k. – obrazę przepisów postępowania, która mogła mieć wpływ na treść orzeczenia, tj.:
1) art. 7 k.p.k. w zw. z art. 410 k.p.k. poprzez dowolną, a nie swobodną ocenę materiału dowodowego i pominięcie wynikających z akt okoliczności przemawiających na korzyść oskarżonego, w szczególności:
   - wyjaśnień oskarżonego z rozprawy z dnia 27 lutego 2025 r., w których podał, że przed przyjazdem na cmentarz był trzeź

In [12]:
from src.output import save_appeal

path = save_appeal(appeal, "baseline")
print("Zapisano apelację do:", path)

Zapisano apelację do: /Users/ewasuknarowska/Projects/WomenInTech/data/output/apelacja_baseline_2026-06-06_155720.txt


## (Opcjonalnie) Wczytaj zapisaną apelację

Zamiast generować od nowa (sekcja 2 — płatne wywołanie LLM), wczytaj **ostatnią**
zapisaną apelację z `data/output/`. Potem możesz puścić same sekcje 3–4 (ewaluacja).

> Do halucynacji (sekcja 4) potrzebny jest jeszcze `docs` — odpal sekcję 1
> (`load_all`, bez kosztu). Do samego pokrycia (sekcja 3) wystarczy `appeal`.

In [3]:
from src.output import load_latest_appeal, latest_appeal_path

appeal = load_latest_appeal("baseline")
print("Wczytano apelację z:", latest_appeal_path("baseline"))
print(f"{len(appeal):,} znaków")

Wczytano apelację z: /Users/ewasuknarowska/Projects/WomenInTech/data/output/apelacja_baseline_2026-06-06_155720.txt
7,911 znaków


## 3. Pokrycie — czy porusza wymagane zagadnienia?

Sędzia-LLM ocenia względem `data/eval.json`.

In [4]:
from src.eval.coverage import evaluate

cov = evaluate(appeal, print_results=True)

[1/12] ✅ 1. W zakresie czynu z art. 178a § 1 k.k. (zarzut pkt I z aktu oskarżenia) istotnym uchybieniem, jaki
     → Apelacja wyraźnie podnosi zarzut błędu w ustaleniach faktycznych co do czynu z art. 178a § 1 k.k., wskazując, że oskarżony miał być nietrzeźwy już o godz. 8.30 przy przyjeździe na cmentarz, mimo braku dowodu na tę okoliczność. Wprost akcentuje też brak przyznania się oskarżonego do tej okoliczności oraz brak podstaw dowodowych dla takiego ustalenia. Zagadnienie z klucza zostało więc faktycznie poruszone.

[2/12] ✅ 1. W zakresie czynu z art. 178a § 1 k.k. (zarzut pkt I z aktu oskarżenia) istotnym uchybieniem, jaki
     → Apelacja wyraźnie podnosi zarzut błędu w ustaleniach faktycznych co do czynu z art. 178a § 1 k.k., wskazując, że oskarżony miał być nietrzeźwy już przy przyjeździe na cmentarz bez podstawy dowodowej. Wprost przywołuje też zeznania świadka Karola Kota, podnosząc, że nie widział on momentu wjazdu ani sposobu znalezienia się pojazdu w alejce cmentarnej. To o

## 3b. Powtarzalność oceny + koszt

Uruchamiamy ocenę pokrycia **5 razy** na tej samej apelacji — czy sędzia jest
powtarzalny (temperatura 0, ale LLM bywa zmienny)? Przy okazji liczymy **koszt**
(tokeny × cennik) i koszt pojedynczego wywołania.

In [5]:
import statistics

from src.eval.coverage import evaluate
from src.llm import track_usage
from src.cost import estimate_cost

model = os.environ["LLM_MODEL"]
scores, costs, calls = [], [], 0
for i in range(1, 6):
    with track_usage() as u:
        cov = evaluate(appeal)  # 5 przebiegów na tej samej apelacji (bez druku)
    scores.append(cov.score)
    costs.append(estimate_cost(u, model))
    calls = u.calls
    print(
        f"Przebieg {i}: {cov.covered}/{cov.total} = {cov.score:.0%} "
        f"| {u.calls} wywołań | {u.total_tokens:,} tok | ${costs[-1]:.4f}"
    )

avg = statistics.mean(costs)
print(
    f"\nPokrycie — średnia {statistics.mean(scores):.0%} "
    f"(min {min(scores):.0%}, max {max(scores):.0%}, odch. {statistics.pstdev(scores):.1%})"
)
print(f"Koszt 1 przebiegu (~{calls} wywołań): ~${avg:.4f}")
print(f"Koszt 1 wywołania: ~${avg / calls:.5f}" if calls else "brak wywołań")

Przebieg 1: 6/12 = 50% | 12 wywołań | 41,070 tok | $0.0385
Przebieg 2: 5/12 = 42% | 12 wywołań | 41,120 tok | $0.0387
Przebieg 3: 6/12 = 50% | 12 wywołań | 41,137 tok | $0.0388
Przebieg 4: 6/12 = 50% | 12 wywołań | 41,113 tok | $0.0387
Przebieg 5: 5/12 = 42% | 12 wywołań | 41,041 tok | $0.0384

Pokrycie — średnia 47% (min 42%, max 50%, odch. 4.1%)
Koszt 1 przebiegu (~12 wywołań): ~$0.0386
Koszt 1 wywołania: ~$0.00322


## 4. Halucynacje — czy apelacja nie zmyśla faktów? (etapowo)

Zamiast jednym wywołaniem (`evaluate_grounding`), pokazujemy to **krok po kroku**:

1. **twierdzenia** wyciągnięte z apelacji,
2. **wybór dokumentów** do sprawdzenia każdego twierdzenia (po opisach akt),
3. **weryfikacja** — czy twierdzenie ma oparcie w wybranych dokumentach.

> Wymaga `docs` z sekcji 1 (`load_all`).

In [6]:
# Krok 1 — twierdzenia faktyczne wyciągnięte z apelacji
from src.eval.grounding import extract_claims

claims = extract_claims(appeal)
print(f"Wyciągnięto {len(claims)} twierdzeń:\n")
for i, c in enumerate(claims, 1):
    print(f"{i:2}. {c.claim}")

Wyciągnięto 40 twierdzeń:

 1. Wyrok Sądu Rejonowego dla Krakowa–Krowodrzy w Krakowie został wydany 14 marca 2025 r. w sprawie o sygnaturze II K 25/25.
 2. Oskarżonym w sprawie jest Daniel Dzik.
 3. Obrońcą Daniela Dzika jest radca prawny Lidia Lis.
 4. Daniel Dzik złożył wyjaśnienia na rozprawie 27 lutego 2025 r.
 5. Daniel Dzik podał na rozprawie 27 lutego 2025 r., że przed przyjazdem na cmentarz był trzeźwy.
 6. Daniel Dzik podał na rozprawie 27 lutego 2025 r., że alkohol spożywał dopiero na cmentarzu w alejce.
 7. Świadek Karol Kot zeznał, że nie widział, kiedy Daniel Dzik wjechał na teren cmentarza.
 8. Świadek Karol Kot zeznał, że nie widział, w jaki sposób Daniel Dzik wjechał na teren cmentarza.
 9. Świadek Karol Kot zeznał, że nie widział, by Daniel Dzik odjeżdżał.
10. Parafia Rzymskokatolicka pw. Św. Krzysztofa w Szczyglicach odpowiedziała pismem z 1 marca 2025 r.
11. Parafia Rzymskokatolicka pw. Św. Krzysztofa w Szczyglicach wskazała, że brama zachodnia cmentarza pozostaje na

### Krok 2 — wybór dokumentów do każdego twierdzenia

LLM na podstawie **opisów** akt (`file_description`) wskazuje, w których plikach
sprawdzić dany fakt — bez wczytywania wszystkich akt naraz.

In [7]:
from src.descriptions import get_descriptions
from src.eval.grounding import select_sources

described = get_descriptions(docs)
selections = [(claim, select_sources(claim.claim, described)) for claim in claims]

for claim, sel in selections:
    print(f"• {claim.claim}")
    print(f"  → wybrane pliki: {sel.files}\n")

• Wyrok Sądu Rejonowego dla Krakowa–Krowodrzy w Krakowie został wydany 14 marca 2025 r. w sprawie o sygnaturze II K 25/25.
  → wybrane pliki: ['15_Protokół_rozprawy_głównej.pdf', '16_Wyrok.pdf']

• Oskarżonym w sprawie jest Daniel Dzik.
  → wybrane pliki: ['08_Postanowienie_o_przedstawieniu_zarzutów.pdf', '11_Akt_oskarżenia.pdf', '16_Wyrok.pdf']

• Obrońcą Daniela Dzika jest radca prawny Lidia Lis.
  → wybrane pliki: ['10_Protokół_przesłuchania_podejrzanego.pdf', '12_Protokół_rozprawy_głównej.pdf', '17_Wniosek_o_sporządzenie_uzasadnienia.pdf']

• Daniel Dzik złożył wyjaśnienia na rozprawie 27 lutego 2025 r.
  → wybrane pliki: ['12_Protokół_rozprawy_głównej.pdf', '15_Protokół_rozprawy_głównej.pdf']

• Daniel Dzik podał na rozprawie 27 lutego 2025 r., że przed przyjazdem na cmentarz był trzeźwy.
  → wybrane pliki: ['12_Protokół_rozprawy_głównej.pdf', '09_Protokół_przesłuchania_podejrzanego.pdf', '10_Protokół_przesłuchania_podejrzanego.pdf']

• Daniel Dzik podał na rozprawie 27 lutego 202

### Krok 3 — weryfikacja: czy twierdzenie to halucynacja?

Każde twierdzenie sprawdzamy **tylko** w wybranych dokumentach:
`supported` (potwierdzone) / `unsupported` (brak oparcia) / `contradicted` (sprzeczne).

In [8]:
from collections import Counter

from src.eval.grounding import verify_claim
from src.sources import prepare_input_texts

ICON = {"supported": "✅", "unsupported": "⚠️", "contradicted": "❌"}
statuses = []
for claim, sel in selections:
    sources_text = prepare_input_texts(docs, sel.files)
    v = verify_claim(claim.claim, sources_text)
    statuses.append(v.status)
    print(f"{ICON.get(v.status, '?')} [{v.status}] {claim.claim}")
    print(f"   → {v.reasoning}\n")

counts = Counter(statuses)
n = len(statuses)
rate = (counts["unsupported"] + counts["contradicted"]) / n if n else 0
print(f"Podsumowanie: {dict(counts)} | wskaźnik halucynacji: {rate:.0%}")

✅ [supported] Wyrok Sądu Rejonowego dla Krakowa–Krowodrzy w Krakowie został wydany 14 marca 2025 r. w sprawie o sygnaturze II K 25/25.
   → Akta wprost wskazują, że wyrok został ogłoszony 14 marca 2025 r. w sprawie o sygnaturze II K 25/25 przez Sąd Rejonowy dla Krakowa–Krowodrzy w Krakowie.

⚠️ [unsupported] Oskarżonym w sprawie jest Daniel Dzik.
   → W aktach Daniel Dzik występuje jako oskarżony, ale nie ma informacji, że jest „oskarżonym w sprawie” w sensie jedynej osoby; fragmenty pokazują jedynie, że to on jest oskarżony w tej konkretnej sprawie. Twierdzenie jest zbyt ogólne i nie wynika wprost z akt jako odrębna teza.

⚠️ [unsupported] Obrońcą Daniela Dzika jest radca prawny Lidia Lis.
   → W aktach wskazano, że obrońcą z wyboru była radca prawny Gabriela Gil, a w późniejszym wniosku o uzasadnienie występuje radca prawny Lidia Lis. Nie ma jednak fragmentu wprost stwierdzającego, że Lidia Lis była obrońcą Daniela Dzika w czasie objętym twierdzeniem.

✅ [supported] Daniel Dzik złoży

## Podsumowanie

Baseline = **1 wywołanie LLM** z ogromnym promptem (patrz tokeny w sekcji 1). Szybki do napisania, ale: zapchany kontekst, „lost in the middle", brak śladu rozumowania i podatność na pominięcia/halucynacje.